# ODI to Databricks Migration

## Target: `HR.TRG_EMP` (Full Load)

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook performs a full load insertion of data from `HR.EMPLOYEES` into the `HR.TRG_EMP` target table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

# ETL Parameters

The following parameters control the execution of this ETL process.

In [ ]:
%sql
-- No specific ETL parameter views are required for this simple full load. 
-- If parameters like `v_ETL_LAST_EXTRACT_TIME` were used, they would be defined here.

In [ ]:
print(f"ETL_JOB_TYPE: {dbutils.widgets.get('ETL_JOB_TYPE')}")
print(f"DATASOURCE_NUM_ID: {dbutils.widgets.get('DATASOURCE_NUM_ID')}")
print(f"ETL_PROC_WID: {dbutils.widgets.get('ETL_PROC_WID')}")
print(f"ODI_SESS_NO: {dbutils.widgets.get('ODI_SESS_NO')}")

# Target Table Insertion

This step performs a full load insertion into the `trg_emp` table from the `employees` source table.

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT 
  INTO workspace.hr.trg_emp
  (
    EMPLOYEE_ID ,
    FIRST_NAME ,
    LAST_NAME ,
    EMAIL ,
    PHONE_NUMBER ,
    HIRE_DATE ,
    JOB_ID ,
    SALARY ,
    COMMISSION_PCT ,
    MANAGER_ID ,
    DEPARTMENT_ID 
  ) 
SELECT 
  employees.EMPLOYEE_ID ,
  employees.FIRST_NAME ,
  employees.LAST_NAME ,
  employees.EMAIL ,
  employees.PHONE_NUMBER ,
  employees.HIRE_DATE ,
  employees.JOB_ID ,
  employees.SALARY ,
  employees.COMMISSION_PCT ,
  employees.MANAGER_ID ,
  employees.DEPARTMENT_ID  
FROM 
  workspace.hr.employees AS employees;

# Validation

Review the number of records inserted into the target table.

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp
FROM workspace.hr.trg_emp;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All schema and table names have been converted to lowercase and prefixed with `workspace.`, e.g., `HR.TRG_EMP` -> `workspace.hr.trg_emp`.
2.  **Oracle Hints:** The Oracle-specific hint `/*+ APPEND PARALLEL */` has been removed as it is not applicable in Databricks Spark SQL.
3.  **Data Types:** The DDL for `TRG_EMP` was not provided in the source. Ensure that the `workspace.hr.trg_emp` Delta table has compatible data types defined for its columns (e.g., `NUMBER` -> `BIGINT`/`DECIMAL`, `DATE` -> `TIMESTAMP`, `VARCHAR2` -> `STRING`). This conversion assumes `TRG_EMP` already exists with an appropriate schema.
4.  **Error Handling:** No explicit error handling (E$ tables) was present in the original task, so none has been added in this conversion. If robust error handling is required, it must be implemented separately.
5.  **Incremental Load:** This task represents a full load. If incremental logic is needed in the future, a `MERGE INTO` statement with `WHEN NOT MATCHED` for inserts would be more appropriate.